<a href="https://colab.research.google.com/github/JoaoAntoni07/estudo-catijr/blob/main/semana-06/semana06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Semana 06

## Algoritmos não supervisionados

### K-Means

#### Carregando os dados e importando as bibliotecas

In [3]:
# importando as bibliotecas necessárias

# EDA e vizualização de dados
import pandas as pd
import plotly.express as px
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# ML
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, pairwise_distances
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer

# Otimização
import optuna

In [4]:
# importando o dataset
from google.colab import drive
drive.mount('/content/drive')
path = '/content/drive/MyDrive/dataset_clientes.csv'
df_clientes = pd.read_csv(path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
df_clientes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   atividade_economica     500 non-null    object 
 1   faturamento_mensal      500 non-null    float64
 2   numero_de_funcionarios  500 non-null    int64  
 3   localizacao             500 non-null    object 
 4   idade                   500 non-null    int64  
 5   inovacao                500 non-null    int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 23.6+ KB


In [10]:
df_clientes.head(10)

,atividade_economica,faturamento_mensal,numero_de_funcionarios,localizacao,idade,inovacao
0,Comércio,713109.95,12,Rio de Janeiro,6,1
1,Comércio,790714.38,9,São Paulo,15,0
2,Comércio,1197239.33,17,São Paulo,4,9
3,Indústria,449185.78,15,São Paulo,6,0
4,Agronegócio,1006373.16,15,São Paulo,15,8
5,Serviços,1629562.41,16,Rio de Janeiro,11,4
6,Serviços,771179.95,13,Vitória,0,1
7,Serviços,707837.61,16,São Paulo,10,6
8,Comércio,888983.66,17,Belo Horizonte,10,1
9,Indústria,1098512.64,13,Rio de Janeiro,9,3


#### EDA

In [11]:
# distribuição da variável inovação
percentual_inovacao = df_clientes.value_counts('inovacao') / len(df_clientes) * 100
px.bar(percentual_inovacao, color=percentual_inovacao.index)

In [ ]:
# teste ANOVA
# verifica se há variações significativas na média de faturamento mensal para diferentes níveis de inovação
# suposições:
# -observações independentes
# -variável dependente é contínua
# -seguem uma distribuição normal
# -homogeneidade das variências
# -amostras sejam de tamanhos iguais

In [14]:
# checar se as variancias (faturamento) entre grupos (inovação) são homogêneas
# aplicar teste de bartlett
# H0 - variâncias são iguais
# H1 - variâncias não são iguais

from scipy.stats import bartlett

# separando os dados de faturamento em grupos coim base na coluna inovação
dados_agrupados = [df_clientes['faturamento_mensal'][df_clientes['inovacao'] == grupo] for grupo in df_clientes['inovacao'].unique()]

# executar o teste de bartlett
bartlett_test_statistic, barlett_p_value = bartlett(*dados_agrupados)

# exibindo resultados
print(f'Estatística de Bartlett: {bartlett_test_statistic}')
print(f'P-Value de Bartlett: {barlett_p_value}')

Estatística de Bartlett: 10.901203117231173
P-Value de Bartlett: 0.28254182954905804


In [15]:
# executar o teste de Shapiro-Wilk
# verificar se os dados seguem uma distribuição normal

from scipy.stats import shapiro

# executar o teste
shapiro_test_statistic, shapiro_p_value = shapiro(df_clientes['faturamento_mensal'])

# exibindo resultados
print(f'Estatística de SW: {shapiro_test_statistic}')
print(f'P-Value de SW: {shapiro_p_value}')

Estatística de SW: 0.9959857602472711
P-Value de SW: 0.23513451034389005


In [6]:
# aplicar a ANOVA de Welch, pois as amostras são de tamanhos diferentes
# H0 - não há diferenças significativas entre as médias dos grupos
# H1 - há pelo menos uma difderença significativa entre as médias dos grupos
from pingouin import welch_anova

aov = welch_anova(dv='faturamento_mensal', between='inovacao', data=df_clientes)

# Exibindo as colunas do DataFrame para diagnóstico
print("Colunas do DataFrame aov:", aov.columns)

# exibindo os resultados
print(f'Estatística do teste de ANOVA Welch: {aov.loc[0, 'F']}')
print(f'P-Value de teste de ANOVA Welch: {aov.loc[0, 'p-unc']}')

Colunas do DataFrame aov: Index(['Source', 'ddof1', 'ddof2', 'F', 'p_unc', 'np2'], dtype='object')
Estatística do teste de ANOVA Welch: 1.1269836194061693


KeyError: 'p-unc'